# Spatial Weights and the `Graph`

Formal spatial models need a way to say which observations are related to one another. In spatial analysis and econometrics, the data structure that holds those relationships is canonically called a spatial weights matrix, $W$, and it specifies the interaction structure we hypothesize among spatial units: a weight $w_{ij}$ expresses a geographical relationship between locations $i$ and $j$. The relationships form a special kind of network graph embedded in geographic space, and `libpysal`'s `Graph` class lets us build, inspect, and transform them.

`Graph` arrived in `libpysal` 4.9 and replaces the older `W` class. Every analysis this afternoon takes a `Graph`, and so does the rest of the morning.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from libpysal.graph import Graph

The graph identifies observations by the index of the dataframe it is built from, so when the data carry a meaningful identifier, such as a census tract's FIPS code, we set it as the index. Distances are measured in the units of the coordinate reference system, so we also project the data, here to UTM zone 11N, whose units are meters.

In [ ]:
df = gpd.read_parquet("data/scag_region.parquet").to_crs(26911).set_index("geoid")
df.shape

The data hold the 4,580 census tracts of the Southern California Association of Governments region. A graph is built from a dataframe with one of the `Graph.build_*` methods.

## Contiguity

A contiguity graph relates observations that share a border. These relationships are conceptually simple and natural for polygons, and since much of urban and regional data comes in polygons (administrative units, census geographies), contiguity graphs are among the most common in practice.

### Rook

Under the rook rule, two polygons are neighbors when they share an edge.

In [ ]:
rook = Graph.build_contiguity(df)
rook.n

The `adjacency` attribute holds the relationships as a pandas series with a two-level index, the focal observation and its neighbor, whose values are the weights.

In [ ]:
rook.adjacency.head()

Slicing the graph with an observation's identifier returns its neighbors and their weights.

In [ ]:
tract = df.index[155]
rook[tract]

`cardinalities` counts each observation's neighbors. In graph terms it is the degree of each node.

In [ ]:
rook.cardinalities.describe()

In [ ]:
rook.cardinalities.hist(bins=range(25));

`pct_nonzero` is the percentage of entries in the $n \times n$ matrix that are not zero. Fewer than one percent are, so this is a very sparse graph, and `sparse` holds it as a scipy sparse matrix whose rows and columns follow the order of `unique_ids`, the dataframe's index.

In [ ]:
rook.pct_nonzero

In [ ]:
rook.sparse

In [ ]:
rook.unique_ids.equals(df.index)

Spatial graphs are sparse because each observation is influenced directly only by observations near it. But neighbors have neighbors of their own, so an influence can travel far by traversing the graph. The `plot` method draws each observation as a node and each relationship as an edge between nodes.

In [ ]:
ax = rook.plot(
    df, node_kws=dict(s=1, alpha=0.3), edge_kws=dict(alpha=0.3, linewidth=0.3), figsize=(8, 8)
)
ax.set_axis_off()

`focal` draws only part of the graph: here, our tract and its neighbors.

In [ ]:
ax = df.loc[[tract] + rook[tract].index.tolist()].plot(alpha=0.3, edgecolor="white", figsize=(5, 5))
rook.plot(df, focal=tract, ax=ax)
ax.set_axis_off()

Not every tract is connected to the rest. A graph's components are its connected pieces, and an isolate is an observation with no neighbors at all.

In [ ]:
rook.n_components, rook.isolates

In [ ]:
rook.component_labels.value_counts()

The region's islands make up the small components: an isolated tract on one of the Channel Islands and the two tracts of Catalina. A disconnected graph has consequences later. No shock in one component can reach another, and some spatial models have trouble with observations that have no neighbors.

### Queen

The queen rule also counts polygons that touch at a single vertex, so it produces more neighbors and a denser graph.

In [ ]:
queen = Graph.build_contiguity(df, rook=False)
queen.pct_nonzero, rook.pct_nonzero

In [ ]:
queen.cardinalities.rename("queen").plot(kind="hist", legend=True, alpha=0.6, bins=range(25))
rook.cardinalities.rename("rook").plot(kind="hist", legend=True, alpha=0.6, bins=range(25));

### Bishop

In a bishop graph, polygons are neighbors when they share only a vertex. Since the queen rule admits an edge or a vertex and the rook rule only an edge,

$$\mathcal{Q} = \mathcal{R} \cup \mathcal{B}, \qquad \mathcal{B} = \mathcal{Q} \setminus \mathcal{R},$$

where $\mathcal{Q}$, $\mathcal{R}$, and $\mathcal{B}$ are the sets of neighbor pairs under each rule. There is no bishop constructor, and bishop graphs encode an odd interaction structure that the social sciences rarely use, but the construction shows something more useful: graphs combine with set operations.

In [ ]:
bishop = queen.difference(rook)
bishop.n_edges

## Distance

Distance graphs relate observations by proximity. They are natural for points, whose locations are discrete and whose distances are well defined. For polygons, we need an assumption about where a polygon is, and the usual one is its centroid.

### Distance band

A distance band makes neighbors of all observations within a threshold, measured in the units of the CRS: here, two kilometers.

In [ ]:
band = Graph.build_distance_band(df.centroid, threshold=2000)
band.cardinalities.describe()

In [ ]:
band.n_components, len(band.isolates)

A fixed threshold keeps the meaning of "neighborhood" constant across observations, so dense places get many neighbors and sparse ones few or none. In a region that runs from downtown Los Angeles to the Mojave, two kilometers leaves hundreds of tracts with no neighbors at all.

### K-nearest neighbors

A k-nearest-neighbor graph turns the distance band around: every observation gets exactly $k$ neighbors, however far away they are. The neighborhood adapts to local density, and nothing is left isolated.

In [ ]:
knn = Graph.build_knn(df.centroid, k=8)
knn.cardinalities.describe()

Nearest-neighbor relationships are often asymmetric. The tract nearest to me may have eight tracts closer to it than I am, so I am its neighbor's neighbor without being its neighbor. `asymmetry` lists the pairs that are not reciprocated, and nearly every tract has at least one.

In [ ]:
asymmetric = knn.asymmetry()
asymmetric.index.nunique() / knn.n

<div class="alert alert-success" style="font-size:120%">
<b>Exercise</b>: <br>
Build the same 8-nearest-neighbor graph from the centroids in longitude and latitude, <code>df.centroid.to_crs(4326)</code>, which treats a degree of longitude and a degree of latitude as equal distances. For how many tracts does the neighbor set change?
</div>

In [ ]:
# %load solutions/02_knn.py

### Kernels

The contiguity, distance-band, and nearest-neighbor graphs so far are binary: a relationship exists, with weight one, or it does not. A kernel graph weights each relationship by distance instead, so that nearer observations count for more, which builds the first law of geography into the graph. The kernel is the shape of the decay; a distance band is the uniform kernel.

In [ ]:
kernel = Graph.build_kernel(df.centroid, kernel="triangular", bandwidth=2000)
kernel[tract]

## Block contiguity

A block graph makes every observation a neighbor of every other observation in the same group. Grouping tracts by county, the first five digits of the FIPS code, relates each tract to every tract in its county and to none outside it.

In [ ]:
county = pd.Series(df.index.str[:5], index=df.index)
block = Graph.build_block_contiguity(county)
block.cardinalities.describe()

## Transformations

Many uses of a graph require its weights to be standardized. Row standardization, the most common, rescales each observation's weights to sum to one, so that the spatial lag of a variable is the average of its neighbors' values. The spatial econometrics literature calls this a transform of the weights matrix, and `Graph.transform` keeps the name.

In [ ]:
rook_r = rook.transform("r")
rook_r[tract]

Unlike the `W` class it replaces, a `Graph` is immutable: `transform` returns a new graph and leaves the original untouched, so the result has to be assigned to a variable.

In [ ]:
rook[tract]

`lag` computes the spatial lag of a variable. With row-standardized weights, it is the average of the variable among each tract's neighbors.

In [ ]:
income = df["median_household_income"].fillna(df["median_household_income"].median())
pd.DataFrame({"income": income, "neighbors' income": rook_r.lag(income)}).head()

A few functions in the PySAL ecosystem still expect the older `W` class. `to_W` converts a graph for them, and `Graph.from_W` converts back.

In [ ]:
type(queen.to_W())

<div class="alert alert-success" style="font-size:120%">
<b>Exercise</b>: <br>
1. Which graph is denser, rook contiguity or four nearest neighbors? <br>
2. How many tracts have fewer rook neighbors than four-nearest neighbors? <br>
3. How many tracts have identical neighbors under queen and rook contiguity?
</div>

In [ ]:
# %load solutions/02.py

---

<a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc-sa/4.0/88x31.png" /></a><br /><span xmlns:dct="http://purl.org/dc/terms/" property="dct:title">Spatial Weights and the Graph</span> by Serge Rey and Eli Knaap is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License</a>.